# Credit Card Approval Prediction — EDA & Data Preprocessing

Steps covered in this notebook (matching the project spec):

1. Environment Setup & Package Installation
2. Dataset Collection & Understanding
3. Data Visualization & Analysis
4. Data Preprocessing & Feature Engineering

## 1. Environment Setup & Package Installation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Dataset Collection & Understanding

Loads `application_record.csv` (applicant profiles) and `credit_record.csv`
(monthly payment history). If these don't exist yet, run `generate_dataset.py`
first, or drop in the real Kaggle dataset with the same column names.

In [ ]:
app_df = pd.read_csv('../data/application_record.csv')
credit_df = pd.read_csv('../data/credit_record.csv', dtype={'STATUS': str})

print(app_df.shape, credit_df.shape)
app_df.head()

In [ ]:
app_df.info()
app_df.describe()

### Converting payment history into an approve/decline label

`STATUS` codes 2–5 (60+ days past due) mark an applicant high-risk.

In [ ]:
bad_status = {'2', '3', '4', '5'}
credit_df['IS_BAD_MONTH'] = credit_df['STATUS'].isin(bad_status).astype(int)
target_df = credit_df.groupby('ID')['IS_BAD_MONTH'].max().reset_index()
target_df = target_df.rename(columns={'IS_BAD_MONTH': 'TARGET'})

df = app_df.merge(target_df, on='ID', how='inner')
df['APPROVAL'] = df['TARGET'].map({0: 'Approved', 1: 'Declined'})
df['TARGET'].value_counts(normalize=True)

## 3. Data Visualization & Analysis

In [ ]:
sns.countplot(data=df, x='APPROVAL', hue='APPROVAL', palette='Set2', legend=False)
plt.title('Approval vs Decline Counts')
plt.show()

In [ ]:
sns.countplot(data=df, y='EMPLOYMENT_STATUS', hue='APPROVAL', palette='Set2')
plt.title('Approval by Employment Status')
plt.show()

In [ ]:
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1) if 'DAYS_BIRTH' in df.columns else df['AGE']
sns.histplot(data=df, x='CREDIT_SCORE', hue='APPROVAL', bins=40, kde=True, palette='Set2', element='step')
plt.title('CIBIL Score Distribution by Approval Outcome')
plt.show()

In [ ]:
sns.countplot(data=df, y='EDUCATION_TYPE', hue='APPROVAL', palette='Set2')
plt.title('Approval by Education Level')
plt.show()

## 4. Data Preprocessing & Feature Engineering

See `preprocessing.py` for the full pipeline (used by both training and the
live Flask app, so both stay perfectly in sync). Summary of what it does:

- Drops duplicate applicant records
- Fills/derives missing values (debt-to-income ratio, etc.)
- Label-encodes categorical fields (gender, education, employment status, family status, housing type)
- Scales numeric fields with `StandardScaler`

In [ ]:
import sys
sys.path.append('..')
from preprocessing import build_dataset

X_scaled, y, encoders, scaler, clean_df = build_dataset(
    app_path='../data/application_record.csv',
    credit_path='../data/credit_record.csv',
)
print('Feature matrix shape:', X_scaled.shape)
print(y.value_counts(normalize=True))